In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
from pathlib import Path
import multiprocessing as mp

# 1. Path Resolution
# Add the parent directory to the system path to import modules
# sys.path.append(str(Path(__file__).absolute().parent))
if 'haman' in os.getcwd():
    sys.path.append('/home/haman/mf-temporary/MeanFieldTester')
    sys.path.append('/home/haman/mf-temporary/MeanFieldTester/codes')
    repo_path = Path('/home/haman/mf-temporary')
elif 'pavel' in os.getcwd():
    sys.path.append('/home/pavel/academia/wintermute/mf-temporary/MeanFieldTester')
    sys.path.append('/home/pavel/academia/wintermute/mf-temporary/MeanFieldTester/codes')
    repo_path = Path('/home/pavel/academia/wintermute/mf-temporary')

# 2. The Great MPI Blindfold
keys_to_delete = [k for k in os.environ if k.startswith('SLURM') or k.startswith('OMPI_') or k.startswith('PRTE_') or 'HYDRA' in k]
for key in keys_to_delete:
    del os.environ[key]

# Prevent CPU hogging by the C++ backend
os.environ["OMP_NUM_THREADS"] = "1" 

# 3. Safe Multiprocessing
try:
    mp.set_start_method('spawn', force=True)
    print("Multiprocessing context set to:", mp.get_start_method())
except RuntimeError:
    pass

print(f"Loaded paths securely. MPI blindfolded. Ready for testing!")

In [ ]:
from importlib import reload

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

from codes.neuron_simulation.config import NeuronSimulationConfig
from codes.transfer_function.config import TransferFunctionConfig
from codes.network_params.loader import load_network_parameters
from codes.plotting import fig_plots
import codes.neuron_simulation as ns
import codes.transfer_function as tf


from pydantic import BaseModel

project_path = repo_path / "projects" / "00_drafts_tests"
os.chdir(project_path)

class TestNeuronSimulationParams(BaseModel):
    neuron_simulation: NeuronSimulationConfig

class TestTransferFunctionParams(BaseModel):
    transfer_function: TransferFunctionConfig

In [ ]:
network_params = load_network_parameters(project_path / "params" / "network_params_new.yaml")
neuron_simulators = ["zerlaut2018", "pynn.nest"]
transfer_functions = ["zerlaut2018", "divolo2019", "neuropsi.custom"]



In [ ]:
# generate test data for transfer function workflow

test_workflow_params = {
    "neuron_simulation": {
        "execution_mode": "run",
        "simulator": "pynn.nest",
        "grid": {
            "exc_neuron" : {
                "grid_type": "adaptive",
                "exc_rate_grid": "adaptive",
                "inh_rate_grid": [0, 30, 10],
                "out_rate_grid": [0, 30, 10],
                "n_coarse_interpolation_points" : 16, 
            },
            "inh_neuron" : {
                "grid_type": "adaptive",
                "exc_rate_grid": "adaptive",
                "inh_rate_grid": [0, 30, 10],
                "out_rate_grid": [0, 30, 10],
                "n_coarse_interpolation_points" : 16, 
            }
        },
        "simulation_time": 10000.0,
        "averaging_window": 8000.0,
        "time_step": 0.1,
        "seed": 42,
        "n_runs": 5,
        "cpus": 16
    }
}

test_config = TestNeuronSimulationParams(**test_workflow_params).neuron_simulation

print("Config loaded successfully!")
print(f"Running for {test_config.simulation_time} ms")

neuron_results = ns.run_neuron_simulation_workflow(test_config, network_params)

In [ ]:
fig_name = f"neuron_activity_dummy_data_linear_grid_{test_config.simulator.replace('.', '_')}.png"
fig_path = project_path / "imgs" / fig_name
fig_plots.fig_neuron_activity(
    neuron_results, 
    common_params={}, 
    fig_params={
        'tight_layout': True,
        'savefig': True,
        'savefig_path': fig_path,
        'title': f"Neuron Activity - {test_config.simulator}"
    }
)
display(Image(filename=str(fig_path)))

fig_name = f"std_neuron_activity_dummy_data_linear_grid_{test_config.simulator.replace('.', '_')}.png"
fig_path = project_path / "imgs" / fig_name
fig_plots.fig_tf_fits_together(
    neuron_results, 
    {"exc_neuron": [],
    "inh_neuron": []
    }, 
    common_params={
        'labels' : [],
        'linestyles' : [],
        # 'xlim' : (0,7),
        'ylim' : (0, 30),
    }, 
    fig_params={
        'fontsize': 14,
        'figsize': (15, 10),  # width, height
        'tight_layout': True,
        'savefig': True,
        'savefig_path': fig_path,
        'title': f"Neuron Activity - {test_config.simulator} (STD)"
    })
display(Image(filename=str(fig_path)))




# Testing transfer function fitting part

### Testing `neuropsi.custom`

In [ ]:
test_tf_workflow_params = {
    "transfer_function": {
        "fit_transfer_function": True,
        "tf_model": {
            "model_name": "neuropsi.custom",
            "square_terms": True,
            "log_term": False,
            "adaptation": True
        },
        "expansion_point": {
            "voltage_mean": -60.0,
            "voltage_std": 4.0,
            "voltage_tau": 0.5
        },
        "expansion_norm": {
            "voltage_mean": 10.0,
            "voltage_std": 6.0,
            "voltage_tau": 1.0
        },
        "fit_with_w": True,
        "out_rate_min": 0.0,
        "out_rate_max": 200.0,
        "V_eff_fitting": {
            "method": "SLSQP",
            "options": {
                "ftol": 5e-9,
                "disp": False,
                "maxiter": 10000
            }
        },
        "TF_fitting": {
            "method": "nelder-mead",
            "options": {
                "fatol": 5e-9,
                "disp": False,
                "maxiter": 10000
            }
        }
    }
}

In [ ]:
tf_params = TestTransferFunctionParams(**test_tf_workflow_params).transfer_function


tf_funcs = tf.run_tf_fitting_workflow(tf_params, network_params, neuron_results)
tf_funcs_legacy = {
    "exc_neuron": [tf_funcs["exc_neuron"]],
    "inh_neuron": [tf_funcs["inh_neuron"]]
}

In [ ]:
fig_name = f"std_neuron_activity_dummy_data_linear_grid_{test_config.simulator.replace('.', '_')}.png"
fig_path = project_path / "imgs" / fig_name
fig_plots.fig_tf_fits_together(
    neuron_results, 
    tf_funcs_legacy, 
    common_params={
        'labels' : ["custom TF"],
        'linestyles' : ['--'],
        # 'xlim' : (0,7),
        'ylim' : (0, 30),
    }, 
    fig_params={
        'fontsize': 14,
        'figsize': (15, 10),  # width, height
        'tight_layout': True,
        'savefig': True,
        'savefig_path': fig_path,
        'title': f"Neuron Activity - {test_config.simulator} (STD)"
    })
display(Image(filename=str(fig_path)))

### Testing `zerlaut2018`

In [ ]:
test_tf_workflow_params = {
    "transfer_function": {
        "fit_transfer_function": True,
        "tf_model": {
            "model_name": "zerlaut2018",
            "square_terms": True,
            "log_term": False
        },
        "expansion_point": {
            "voltage_mean": -60.0,
            "voltage_std": 4.0,
            "voltage_tau": 0.5
        },
        "expansion_norm": {
            "voltage_mean": 10.0,
            "voltage_std": 6.0,
            "voltage_tau": 1.0
        },
        "fit_with_w": True,
        "out_rate_min": 0.0,
        "out_rate_max": 200.0,
        "V_eff_fitting": {
            "method": "SLSQP",
            "options": {
                "ftol": 5e-9,
                "disp": False,
                "maxiter": 10000
            }
        },
        "TF_fitting": {
            "method": "nelder-mead",
            "options": {
                "fatol": 5e-9,
                "disp": False,
                "maxiter": 10000
            }
        }
    }
}

In [ ]:
tf_params = TestTransferFunctionParams(**test_tf_workflow_params).transfer_function


tf_funcs = tf.run_tf_fitting_workflow(tf_params, network_params, neuron_results)
tf_funcs_legacy = {
    "exc_neuron": [tf_funcs["exc_neuron"]],
    "inh_neuron": [tf_funcs["inh_neuron"]]
}

In [ ]:
fig_name = f"std_neuron_activity_dummy_data_linear_grid_{test_config.simulator.replace('.', '_')}.png"
fig_path = project_path / "imgs" / fig_name
fig_plots.fig_tf_fits_together(
    neuron_results, 
    tf_funcs_legacy, 
    common_params={
        'labels' : ["custom TF"],
        'linestyles' : ['--'],
        # 'xlim' : (0,7),
        'ylim' : (0, 30),
    }, 
    fig_params={
        'fontsize': 14,
        'figsize': (15, 10),  # width, height
        'tight_layout': True,
        'savefig': True,
        'savefig_path': fig_path,
        'title': f"Neuron Activity - {test_config.simulator} (STD)"
    })
display(Image(filename=str(fig_path)))

### Testing `divolo2019`

In [ ]:
test_tf_workflow_params = {
    "transfer_function": {
        "fit_transfer_function": True,
        "tf_model": {
            "model_name": "divolo2019",
            "square_terms": True,
        },
        "expansion_point": {
            "voltage_mean": -60.0,
            "voltage_std": 4.0,
            "voltage_tau": 0.5
        },
        "expansion_norm": {
            "voltage_mean": 10.0,
            "voltage_std": 6.0,
            "voltage_tau": 1.0
        },
        "fit_with_w": True,
        "out_rate_min": 0.0,
        "out_rate_max": 200.0,
        "V_eff_fitting": {
            "method": "SLSQP",
            "options": {
                "ftol": 5e-9,
                "disp": False,
                "maxiter": 10000
            }
        },
        "TF_fitting": {
            "method": "nelder-mead",
            "options": {
                "fatol": 5e-9,
                "disp": False,
                "maxiter": 10000
            }
        }
    }
}

In [ ]:
tf_params = TestTransferFunctionParams(**test_tf_workflow_params).transfer_function


tf_funcs = tf.run_tf_fitting_workflow(tf_params, network_params, neuron_results)
tf_funcs_legacy = {
    "exc_neuron": [tf_funcs["exc_neuron"]],
    "inh_neuron": [tf_funcs["inh_neuron"]]
}

In [ ]:
fig_name = f"std_neuron_activity_dummy_data_linear_grid_{test_config.simulator.replace('.', '_')}.png"
fig_path = project_path / "imgs" / fig_name
fig_plots.fig_tf_fits_together(
    neuron_results, 
    tf_funcs_legacy, 
    common_params={
        'labels' : ["custom TF"],
        'linestyles' : ['--'],
        # 'xlim' : (0,7),
        'ylim' : (0, 30),
    }, 
    fig_params={
        'fontsize': 14,
        'figsize': (15, 10),  # width, height
        'tight_layout': True,
        'savefig': True,
        'savefig_path': fig_path,
        'title': f"Neuron Activity - {test_config.simulator} (STD)"
    })
display(Image(filename=str(fig_path)))